# Lab 2 - Intent Classification Baseline

In this lab, we will develop an intent classification baseline following a simplified **evidence-driven model development process**.

Using a dataset of banking-related customer requests, we will inspect the data, train a baseline classifier, evaluate its performance, and analyze its errors. Based on our observations, we will formulate hypotheses, test a possible improvement, and assess the model's generalization on held-out test data.

The goal is not simply to achieve high accuracy, but to **understand how experimental evidence informs model development decisions**.

### Prerequisites

Before starting the lab, make sure your environment is properly configured.

The repository includes setup scripts in the `/scripts` directory. Run `install_env.sh` to install the additional dependencies required for this lab.

```bash
bash scripts/install_env.sh
```

Finally, make sure your Jupyter notebook is using the **HITL Course** kernel.

## 1. Data exploration (BANKING77)

For our **intent classification** task, we will use **BANKING77**, a dataset of banking-related customer requests available on Hugging Face as `PolyAI/banking77`.

### 1.1 Loading the dataset

If your Jupyter notebook cannot access the Internet (e.g., on the YourAI cluster), download the dataset beforehand using the `download_dataset.py` script:

````bash
python scripts/download_dataset.py "PolyAI/banking77" "data/PolyAI_banking77"
````

In [ ]:
from datasets import load_dataset, load_from_disk

#ds = load_dataset("PolyAI/banking77"
ds = load_from_disk("../../data/PolyAI_banking77")

ds

### 1.2 Inspecting the dataset

Before training a model, we should understand the available data.

BANKING77 contains customer requests labeled with 77 banking-related intents. It provides predefined training and test sets, but no validation set.

Let's inspect:
- The size and structure of the dataset.
- Some example messages and their labels.
- The distribution of intent categories.

**Question:** Why should we inspect the dataset before selecting and training a model?


In [ ]:
import pandas as pd

print(ds)
print(ds["train"].features)

# Labels are stored as integers; retrieve their names
label_names = ds["train"].features["label"].names

print(f"\nNumber of intents: {len(label_names)}")
print("Example intent labels:", label_names[:10])


##### Let's look at some examples

In [ ]:
ds_train = ds["train"]

# we need to map the intents (that are numbers) to the labels
df_train = pd.DataFrame({
    "text": ds_train["text"],
    "intent": [label_names[i] for i in ds_train["label"]]
})

# let's show a sample
display(df_train.sample(5, random_state=42))

In [ ]:
# distribution by intent
# print(df_train["intent"].value_counts().to_string())
print(df_train["intent"].value_counts())

In [ ]:
display(df_train.describe())
df_train.isna().sum()            # Missing values

### 1.3 Selecting our prediction task

BANKING77 contains 77 intents. For this lab, we will focus on **nine related intents** involving account access, transfers, card payments, and cash withdrawals.

These classes include similar types of requests that may be difficult to distinguish.

We will map the original BANKING77 labels to more readable names.

**Question:** Why might distinguishing *pending transfer* from *transfer not received* be more challenging than distinguishing *account access* from *card payment*?


In [ ]:
# Original BANKING77 label -> course label
source_to_intent = {
    "passcode_forgotten": "account_access_passcode",
    "pending_transfer": "bank_transfer_pending",
    "transfer_not_received_by_recipient": "bank_transfer_not_received",
    "pending_card_payment": "card_payment_pending",
    "declined_card_payment": "card_payment_declined",
    "transaction_charged_twice": "card_payment_duplicated",
    "pending_cash_withdrawal": "cash_withdrawal_pending",
    "cash_withdrawal_not_recognised": "cash_withdrawal_unrecognized",
    "card_acceptance" : "card_acceptance"
}

# Original BANKING77 intents we want to keep
target_intents = list(source_to_intent.keys())

# Convert the test set to Pandas as well
df_test = ds["test"].to_pandas()
df_test["intent"] = df_test["label"].map(lambda i: label_names[i])

# Filter the datasets
df_train = df[df["intent"].isin(target_intents)].copy()
df_test = df_test[df_test["intent"].isin(target_intents)].copy()

# Rename the intents
df_train["intent"] = df_train["intent"].map(source_to_intent)
df_test["intent"] = df_test["intent"].map(source_to_intent)


print("Training examples:", len(df_train))
print("Test examples:", len(df_test))

In [ ]:
display(df_train["intent"].value_counts())

### 1.4 Creating the validation set

Our dataset contains training and test partitions, but we also need validation data.

Recall their different roles:

- **Training:** Learn model parameters.
- **Validation:** Evaluate candidates, inspect errors, and guide improvements.
- **Test:** Independent evaluation after development is complete.

We will reserve **20% of the selected official training data** for validation.

We use a **stratified split** to approximately preserve the intent distribution across partitions.

The official test set remains reserved for final evaluation.

**Question:** Why shouldn't we use the official test set to decide which model configuration works best?


In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

df_train_candidates, df_val = train_test_split(
    df_train,
    test_size=0.20,
    stratify=df_train["intent"],
    random_state=SEED
)

# Keep the official test set separate
print("Training candidates:", len(df_train_candidates))
print("Validation:", len(df_val))
print("Test (reserved):", len(df_test))

## 2. Establishing the Baseline

Before exploring more sophisticated models, we need a simple baseline against which to compare future improvements.

For our intent classification task, we will use:

- **Representation:** Bag-of-Words (BoW), representing each message through word counts.
- **Classifier:** Logistic Regression, learning to predict one of our nine intents from these features.

We will also include a trivial baseline that always predicts the most frequent intent.

Both models will be trained on the same data and evaluated using the same validation set.


### 2.1 A trivial baseline
A majority-class predictor provides a simple reference. Any learned model should ideally perform better than this.


In [ ]:
from sklearn.dummy import DummyClassifier

# Prepare the supervised input (x, y) for the training
X_train = df_train_candidates["text"]
y_train = df_train_candidates["intent"]

X_val = df_val["text"]
y_val = df_val["intent"]

# Always predict the most frequent training class
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train.to_frame(), y_train)

y_pred_dummy = dummy.predict(X_val.to_frame())

print("Majority class:", dummy.classes_[dummy.class_prior_.argmax()])


### 2.2 Training a Bag-of-Words Classifier

Our first learned model combines two components:

1. **CountVectorizer:** Converts each message into a numerical vector representing word frequencies.
2. **Logistic Regression:** Learns the relationship between these features and the target intents.

We fit the vectorizer and classifier using **training data only**. Validation data is reserved for evaluating the model.

**Question:** What information might be lost when representing a sentence as a collection of word counts?


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

# Convert text into Bag-of-Words features
vectorizer = CountVectorizer()

X_train_bow = vectorizer.fit_transform(X_train)
X_val_bow = vectorizer.transform(X_val)

print("Training matrix:", X_train_bow.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))


**Notice** the distinction between fit_transform() and transform():
- Training: learn the vocabulary and transform the messages.
- Validation: use the existing vocabulary to transform the messages.

In [ ]:
# Train the classifier
model = LogisticRegression(max_iter=1000, random_state=SEED)
model.fit(X_train_bow, y_train)

# Generate predictions on validation data
y_pred = model.predict(X_val_bow)

### 2.3 Inspecting Predictions

Our model can now predict intents for messages it has not seen during training.

Let's inspect a few predictions.

**Questions:**
- Are the predictions plausible?
- Do any mistakes stand out?
- Can we already conclude that the model performs well?


In [ ]:

predictions = pd.DataFrame({
    "text": X_val.values,
    "actual": y_val.values,
    "predicted": y_pred
})

display(predictions.sample(15, random_state=SEED))


## 3. Evaluating the Baseline

We now have two models: a majority-class predictor and our Bag-of-Words classifier.

How well do they perform on unseen validation examples?

We will evaluate them using:

- **Accuracy:** The proportion of correctly classified messages.
- **Macro-F1:** The average F1-score across intents, giving each class equal weight.
- **Class-wise metrics:** Precision, recall, and F1 for each intent.
- **Confusion matrix:** Which intents are being confused with one another?

Remember: We use the **validation set**, not the reserved test set, to guide model development.


### 3.1 Comparing the baselines

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

results = pd.DataFrame({
    "Model": ["Majority class", "BoW + Logistic Regression"],
    "Accuracy": [
        accuracy_score(y_val, y_pred_dummy),
        accuracy_score(y_val, y_pred)
    ],
    "Macro-F1": [
        f1_score(y_val, y_pred_dummy, average="macro"),
        f1_score(y_val, y_pred, average="macro")
    ]
})

display(results.round(3))

**Questions:**

1. How much better is our learned model than the trivial baseline?
2. Would accuracy alone be sufficient to evaluate this classifier?

### 3.2 Class-wise Performance

A model may perform well overall while struggling with particular intents.

Let's examine **precision, recall, F1-score, and support** for each class.

**Support** indicates how many validation examples belong to each intent. This matters because performance estimates based on very few examples can be unstable.


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_val,
    y_pred,
    digits=2,
    zero_division=0
))

**Questions:**

1. Which intents have the lowest F1-scores?
2. Are the problems caused by low precision, low recall, or both?
3. How many validation examples support these results?

A low score does not automatically tell us **why** a model struggles. We need to inspect its mistakes before drawing conclusions.

### 3.3 Confusion Matrix

A confusion matrix shows which classes the model predicts correctly and which classes it confuses.

- **Rows:** Actual intents.
- **Columns:** Predicted intents.
- **Diagonal:** Correct predictions.
- **Off-diagonal:** Misclassifications.

We can use it to identify systematic patterns that aggregate metrics might hide.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(11, 8))

ConfusionMatrixDisplay.from_predictions(
    y_val,
    y_pred,
    ax=ax,
    cmap="Blues",
    values_format="d",
    xticks_rotation=90,
    colorbar=False
)

plt.tight_layout()
plt.show()

**Questions:**

1. Which pairs of intents are most frequently confused?
2. Are the errors concentrated in a few classes or spread across many?
3. Do the confused intents have something in common?

The confusion matrix tells us **where mistakes occur**, but not necessarily **what causes them**.

### 3.4 Interpreting the Results

Based on the validation results, summarize:

1. **Overall performance:** How well does our classifier perform compared with the trivial baseline?
2. **Main limitation:** What is the most important weakness revealed by the evaluation?
3. **Open question:** What do we still need to investigate before deciding how to improve the model?

Distinguish **observations supported by the evaluation** from **possible explanations that still need testing**.


## 4. Error Analysis and Diagnosis

Our evaluation tells us **how well the model performs and which intents it confuses**. But it does not explain why those errors occur.

### 4.1 Inspecting misclassified examples

Let's examine individual misclassified messages to identify patterns and formulate possible explanations.

**Questions:**
- Do the mistakes share common characteristics?
- Are some messages ambiguous or difficult to classify?
- What might explain the observed errors?


In [ ]:
# Inspect all misclassified validation examples
errors = predictions[
    predictions["actual"] != predictions["predicted"]
].copy()

print(f"Misclassified examples: {len(errors)} / {len(predictions)}")

with pd.option_context("display.max_colwidth", None):
    display(errors.sort_values(["actual", "predicted"]))


### 4.2 Investigating an Error Pattern

Choose an intent that appears frequently in the model's mistakes.

Compare incorrectly and correctly classified messages involving that intent.

**Questions:**
- What distinguishes the misclassified messages from the correctly classified ones?
- Are there particular words or expressions associated with the mistakes?
- Could the problem involve training coverage, the text representation, or ambiguous labels?

Remember: These are **possible explanations**, not established causes.

In [ ]:
# Choose an intent to investigate
focus_intent = "card_payment_pending"

# Errors involving this intent (FP, FN)
focus_errors = errors[
    (errors["actual"] == focus_intent) |
    (errors["predicted"] == focus_intent)
]

# Correctly classified examples for comparison (TP)
focus_correct = predictions[
    (predictions["actual"] == focus_intent) &
    (predictions["predicted"] == focus_intent)
].sample(frac=1, random_state=SEED).head(5)

with pd.option_context("display.max_colwidth", None):
    print("Misclassified examples:")
    display(focus_errors)

    print("Correctly classified examples:")
    display(focus_correct)


### 4.3 Formulating Hypotheses

An observed error pattern does not necessarily reveal its cause.

For example, errors involving a particular intent might result from:

- **Training data:** Insufficient examples of certain expressions or situations.
- **Representation:** Bag-of-Words may fail to capture relevant phrases or word order.
- **Task definition:** Some messages may be ambiguous or difficult to label consistently.
- **Model behavior:** The classifier may rely on features that do not reliably distinguish the intents.

**Your task:**

1. **Observation:** Describe one recurring error pattern, using examples as evidence.
2. **Hypothesis:** Propose a possible explanation for that pattern.
3. **Investigation:** What evidence or experiment could help test your hypothesis?

Be specific. "The model needs improvement" is not a hypothesis.

## 5. From Diagnosis to Next Steps

Our error analysis revealed several possible explanations for the model's mistakes. But observations alone do not establish their causes.

How could we investigate these hypotheses?

**1. Investigate model behavior**

Why did the model predict one intent rather than another? Which words or features influenced its decision?

We could inspect the features and coefficients learned by Logistic Regression. Later in the course, we will explore **interpretable ML methods** that help us investigate model behavior more systematically.

**2. Test an intervention: Improve the representation**

Our current Bag-of-Words representation uses raw word counts. Perhaps a different representation could improve the model's ability to distinguish related intents.

One possible experiment is to replace `CountVectorizer` with `TfidfVectorizer`, which weights terms according to their frequency within documents and across the dataset.

To test this hypothesis, we would:

1. Keep the training data, classifier, and validation set unchanged.
2. Train a new model using TF-IDF features.
3. Compare its validation performance against the original baseline.
4. Examine whether the errors we identified have changed.

**Questions:**
- What results would support our hypothesis?
- What would we conclude if the performance remained unchanged?
- Which errors might require additional context rather than a different representation?


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline

# Train the alternative model (TF-IDF + Logistic Regression)
tfidf_model = make_pipeline(
    TfidfVectorizer(),
    LogisticRegression(max_iter=1000, random_state=SEED)
)

tfidf_model.fit(X_train, y_train)
y_pred_tfidf = tfidf_model.predict(X_val)

# Compare validation performance
comparison = pd.DataFrame({
    "Model": ["BoW + Logistic Regression", "TF-IDF + Logistic Regression"],
    "Accuracy": [
        accuracy_score(y_val, y_pred),
        accuracy_score(y_val, y_pred_tfidf)
    ],
    "Macro-F1": [
        f1_score(y_val, y_pred, average="macro"),
        f1_score(y_val, y_pred_tfidf, average="macro")
    ]
})

display(comparison.round(3))

## 6. Final Evaluation

We have developed a baseline, examined its validation performance, and identified possible directions for improvement.

We will now evaluate our **existing baseline model** on the reserved test set.

Unlike the validation set, the test set has not been used to guide our model-development decisions. It provides an independent assessment of performance on held-out examples.

After inspecting the test results, we should not modify the model based on those results and then report performance on the same test set as an independent evaluation.


In [ ]:

# Apply the existing representation and classifier
X_test_bow = vectorizer.transform(df_test["text"])
y_test = df_test["intent"]

y_pred_test = model.predict(X_test_bow)

# Final evaluation
print(classification_report(y_test, y_pred_test, digits=2))


In [ ]:

final_results = pd.DataFrame({
    "Metric": ["Accuracy", "Macro-F1"],
    "Validation": [
        accuracy_score(y_val, y_pred),
        f1_score(y_val, y_pred, average="macro")
    ],
    "Test": [
        accuracy_score(y_test, y_pred_test),
        f1_score(y_test, y_pred_test, average="macro")
    ]
})

display(final_results.round(3))



### Reflecting on the Results

1. How does test performance compare with validation performance?
2. What conclusions can we support about the baseline?
3. What limitations remain unresolved?
4. Which experiment would you prioritize next, and why?

**Looking ahead:** If our model could benefit from additional labeled examples, how should we decide which examples are worth labeling?
